# Assignment 05 · Notebook 00: Khái niệm cơ bản của CNN qua hàm và hợp hàm

**Sinh viên:** Nguyễn Duy Nghĩa · B23DCCN600 · D23CTPM01 · **GVHD:** PGS.TS Trần Đình Quế

Notebook phục vụ **mục 1** của đề. Mỗi khái niệm được viết thành một hàm NumPy trong
`a05/concepts.py`, tính thử trên ví dụ nhỏ, rồi so với PyTorch bằng `np.allclose`.
Ý chính: một CNN chỉ là hợp của các hàm $f = f_L \circ \dots \circ f_2 \circ f_1$.

In [1]:
import sys
sys.path.insert(0, "..")
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn
from a05 import concepts as C
from a05.export import save_metrics, write_dat

rng = np.random.default_rng(42)
checks = {}
print("Thiết bị: CPU (NumPy", np.__version__, "· PyTorch", torch.__version__ + ")")

Thiết bị: CPU (NumPy 2.5.1 · PyTorch 2.13.0+cu126)


## 1. Neuron là một hàm affine
$z = \mathbf{w}^\top \mathbf{x} + b$. So với `nn.Linear` có cùng trọng số.

In [2]:
x, w, b = rng.standard_normal(4), rng.standard_normal(4), 0.5
lin = nn.Linear(4, 1).double()
with torch.no_grad():
    lin.weight[:] = torch.tensor(w)[None]
    lin.bias[:] = b
ours, ref = C.neuron(x, w, b), lin(torch.tensor(x)).item()
checks["neuron"] = bool(np.isclose(ours, ref))
print(f"neuron = {ours:.6f} · nn.Linear = {ref:.6f} · khớp: {checks['neuron']}")

neuron = 1.058224 · nn.Linear = 1.058224 · khớp: True


## 2. Vì sao cần hàm phi tuyến
Hai tầng affine ghép nối tiếp vẫn là **một** tầng affine: $W_2(W_1x+b_1)+b_2 = (W_2W_1)x + (W_2b_1+b_2)$.
Nếu không có ReLU ở giữa, chồng bao nhiêu tầng cũng không tăng sức biểu diễn.

In [3]:
W1, b1 = rng.standard_normal((5, 4)), rng.standard_normal(5)
W2, b2 = rng.standard_normal((3, 5)), rng.standard_normal(3)
two_layers = C.compose(C.affine(W1, b1), C.affine(W2, b2))
W, bb = C.collapse_affine(W1, b1, W2, b2)
checks["collapse_affine"] = bool(np.allclose(two_layers(x), C.affine(W, bb)(x)))
with_relu = C.compose(C.affine(W1, b1), C.relu, C.affine(W2, b2))
print("affine ∘ affine == một affine:", checks["collapse_affine"])
print("affine ∘ ReLU ∘ affine == một affine:", bool(np.allclose(with_relu(x), C.affine(W, bb)(x))))

affine ∘ affine == một affine: True
affine ∘ ReLU ∘ affine == một affine: False


## 3. Hàm kích hoạt và đạo hàm
Dữ liệu xuất ra `act.dat` để vẽ hình bằng pgfplots.

In [4]:
z = np.linspace(-4, 4, 161)
write_dat("act", {"z": z, "relu": C.relu(z), "drelu": C.relu_grad(z), "sigmoid": C.sigmoid(z),
                  "dsigmoid": C.sigmoid_grad(z), "tanh": C.tanh(z), "dtanh": C.tanh_grad(z)})
zt = torch.tensor(z, requires_grad=True)
for name, f, g, tf in [("relu", C.relu, C.relu_grad, torch.relu),
                       ("sigmoid", C.sigmoid, C.sigmoid_grad, torch.sigmoid),
                       ("tanh", C.tanh, C.tanh_grad, torch.tanh)]:
    y = tf(zt)
    (grad,) = torch.autograd.grad(y.sum(), zt)
    mask = np.abs(z) > 1e-9  # ReLU không khả vi tại 0
    checks[name] = bool(np.allclose(f(z), y.detach().numpy()) and np.allclose(g(z)[mask], grad.numpy()[mask]))
    print(f"{name:8s} giá trị và đạo hàm khớp autograd: {checks[name]}")

relu     giá trị và đạo hàm khớp autograd: True
sigmoid  giá trị và đạo hàm khớp autograd: True
tanh     giá trị và đạo hàm khớp autograd: True


## 4. Tích chập 2D tính tay
Ảnh 5×5, bộ lọc 3×3, stride 1, không đệm: đầu ra có kích thước $(5-3)/1+1 = 3$. Ô đầu ra $(0,0)$ là
tổng tích từng phần tử của bộ lọc với góc trên trái của ảnh.

In [5]:
Xs = rng.integers(0, 4, size=(5, 5)).astype(float)
Ks = np.array([[1., 0, -1], [1, 0, -1], [1, 0, -1]])
Ys = C.conv2d(Xs[None], Ks[None, None])[0]
print("X =\n", Xs, "\nK =\n", Ks, "\nY = conv2d(X, K) =\n", Ys)
terms = " + ".join(f"{a:g}·{k:g}" for a, k in zip(Xs[:3, :3].ravel(), Ks.ravel()))
print("Ô (0,0) tính tay:", terms, "=", Ys[0, 0])

X =
 [[3. 2. 1. 2. 1.]
 [3. 2. 1. 2. 2.]
 [0. 0. 0. 0. 1.]
 [2. 2. 1. 3. 2.]
 [0. 3. 2. 2. 2.]] 
K =
 [[ 1.  0. -1.]
 [ 1.  0. -1.]
 [ 1.  0. -1.]] 
Y = conv2d(X, K) =
 [[ 4.  0. -2.]
 [ 3. -1. -3.]
 [-1.  0. -2.]]
Ô (0,0) tính tay: 3·1 + 2·0 + 1·-1 + 3·1 + 2·0 + 1·-1 + 0·1 + 0·0 + 0·-1 = 4.0


In [6]:
Xr, Kr, br = rng.standard_normal((3, 9, 9)), rng.standard_normal((4, 3, 3, 3)), rng.standard_normal(4)
for s, p in [(1, 0), (1, 1), (2, 1)]:
    ours = C.conv2d(Xr, Kr, br, stride=s, pad=p)
    ref = F.conv2d(torch.tensor(Xr)[None], torch.tensor(Kr), torch.tensor(br), stride=s, padding=p)[0].numpy()
    key = f"conv2d_s{s}_p{p}"
    checks[key] = bool(np.allclose(ours, ref))
    print(f"stride {s}, pad {p}: shape {ours.shape}, out_size = {C.out_size(9, 3, p, s)}, khớp F.conv2d: {checks[key]}")

stride 1, pad 0: shape (4, 7, 7), out_size = 7, khớp F.conv2d: True
stride 1, pad 1: shape (4, 9, 9), out_size = 9, khớp F.conv2d: True
stride 2, pad 1: shape (4, 5, 5), out_size = 5, khớp F.conv2d: True


## 5. Số tham số: Dense so với Conv
Ảnh 32×32×3 nối đầy đủ tới 64 đơn vị so với một tầng Conv 3×3 ra 64 kênh.

In [7]:
dense_p, conv_p = C.count_params_dense(32 * 32 * 3, 64), C.count_params_conv(3, 64, 3)
checks["count_params"] = (dense_p == sum(q.numel() for q in nn.Linear(3072, 64).parameters())
                          and conv_p == sum(q.numel() for q in nn.Conv2d(3, 64, 3).parameters()))
print(f"Dense 3072→64: {dense_p:,} tham số · Conv 3×3, 3→64: {conv_p:,} tham số · "
      f"tỉ lệ {dense_p / conv_p:.0f} lần · khớp numel: {checks['count_params']}")

Dense 3072→64: 196,672 tham số · Conv 3×3, 3→64: 1,792 tham số · tỉ lệ 110 lần · khớp numel: True

## 6. Max pooling

In [8]:
Xp = rng.integers(0, 10, size=(4, 4)).astype(float)
Yp = C.maxpool2d(Xp[None])[0]
checks["maxpool"] = bool(np.allclose(C.maxpool2d(Xr), F.max_pool2d(torch.tensor(Xr)[None], 2)[0].numpy()))
print("X =\n", Xp, "\nmaxpool 2×2 =\n", Yp, "\nkhớp F.max_pool2d:", checks["maxpool"])

X =


 [[5. 1. 1. 6.]
 [5. 5. 7. 5.]
 [6. 0. 8. 4.]
 [1. 8. 7. 0.]] 
maxpool 2×2 =
 [[5. 7.]
 [8. 8.]] 
khớp F.max_pool2d: True


## 7. Softmax và cross-entropy ổn định số
Với logits rất lớn, $e^{1000}$ tràn số; trừ đi $\max z$ trước khi lấy mũ không đổi kết quả.

In [9]:
zl = np.array([1000., 1001., 1002.])
print("softmax([1000, 1001, 1002]) =", C.softmax(zl).round(6))
zc = rng.standard_normal(10)
checks["cross_entropy"] = bool(np.isclose(C.cross_entropy(zc, 3),
                                          F.cross_entropy(torch.tensor(zc)[None], torch.tensor([3])).item()))
checks["softmax"] = bool(np.allclose(C.softmax(zc), torch.softmax(torch.tensor(zc), 0).numpy()))
print("cross_entropy khớp F.cross_entropy:", checks["cross_entropy"], "· softmax khớp:", checks["softmax"])

softmax([1000, 1001, 1002]) = [0.090031 0.244728 0.665241]
cross_entropy khớp F.cross_entropy: True · softmax khớp: True


## 8. CNN nhỏ là một hợp hàm
$f = \mathrm{dense} \circ \mathrm{flatten} \circ \mathrm{maxpool} \circ \mathrm{ReLU} \circ \mathrm{conv}$,
chạy trên ảnh 3×8×8. In shape sau từng hàm, rồi so với `nn.Sequential` có cùng trọng số.

In [10]:
p = {"K": rng.standard_normal((4, 3, 3, 3)) * 0.3, "b": np.zeros(4),
     "W": rng.standard_normal((10, 64)) * 0.1, "c": np.zeros(10)}
X8 = rng.standard_normal((3, 8, 8))
logits, trace = C.tiny_cnn_forward(X8, p)
for name, shape in trace:
    print(f"{name:8s} -> {shape}")
seq = nn.Sequential(nn.Conv2d(3, 4, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2), nn.Flatten(), nn.Linear(64, 10)).double()
with torch.no_grad():
    seq[0].weight[:], seq[0].bias[:] = torch.tensor(p["K"]), torch.tensor(p["b"])
    seq[4].weight[:], seq[4].bias[:] = torch.tensor(p["W"]), torch.tensor(p["c"])
checks["tiny_cnn"] = bool(np.allclose(logits, seq(torch.tensor(X8)[None])[0].detach().numpy()))
print("logits khớp nn.Sequential:", checks["tiny_cnn"])

input    -> (3, 8, 8)
conv     -> (4, 8, 8)
relu     -> (4, 8, 8)
maxpool  -> (4, 4, 4)
flatten  -> (64,)
dense    -> (10,)
logits khớp nn.Sequential: True


## 9. Lan truyền ngược là quy tắc chuỗi
Với $L = \sum Y \odot G$ ($G$ là gradient từ tầng sau), so gradient giải tích của `conv2d_backward`
với sai phân hữu hạn trung tâm. Sai số tương đối phải nhỏ hơn $10^{-6}$.

In [11]:
G = rng.standard_normal(C.conv2d(Xr, Kr, br, 2, 1).shape)
dX, dK, db = C.conv2d_backward(Xr, Kr, G, stride=2, pad=1)
err_K = C.grad_check(lambda K: (C.conv2d(Xr, K, br, 2, 1) * G).sum(), Kr.copy(), dK)
err_X = C.grad_check(lambda X: (C.conv2d(X, Kr, br, 2, 1) * G).sum(), Xr.copy(), dX)
err_b = C.grad_check(lambda b_: (C.conv2d(Xr, Kr, b_, 2, 1) * G).sum(), br.copy(), db)
print(f"sai số tương đối: dK {err_K:.2e} · dX {err_X:.2e} · db {err_b:.2e}")
checks["grad_check"] = bool(max(err_K, err_X, err_b) < 1e-6)
eta, L0 = 1e-3, (C.conv2d(Xr, Kr, br, 2, 1) * G).sum()
L1 = (C.conv2d(Xr, Kr - eta * dK, br, 2, 1) * G).sum()
print(f"L trước {L0:.4f} → sau một bước θ ← θ − η∇L: {L1:.4f}")

sai số tương đối: dK 5.34e-10 · dX 6.64e-10 · db 7.16e-10
L trước 45.9464 → sau một bước θ ← θ − η∇L: 43.7258


In [12]:
print("Tất cả phép kiểm chứng:", checks)
assert all(checks.values())
save_metrics("concepts", {
    "checks_passed": sum(checks.values()), "checks_total": len(checks),
    "grad_err": {"K": err_K, "X": err_X, "b": err_b},
    "gd_step": {"before": L0, "after": L1, "eta": eta},
    "params": {"dense": dense_p, "conv": conv_p, "ratio": dense_p / conv_p},
    "conv_example": {"X": Xs.astype(int), "K": Ks.astype(int), "Y": Ys.astype(int)},
    "pool_example": {"X": Xp.astype(int), "Y": Yp.astype(int)},
    "trace": [[n, "×".join(map(str, s))] for n, s in trace],
})

Tất cả phép kiểm chứng: {'neuron': True, 'collapse_affine': True, 'relu': True, 'sigmoid': True, 'tanh': True, 'conv2d_s1_p0': True, 'conv2d_s1_p1': True, 'conv2d_s2_p1': True, 'count_params': True, 'maxpool': True, 'cross_entropy': True, 'softmax': True, 'tiny_cnn': True, 'grad_check': True}


WindowsPath('D:/Python/Intelligent-System-Development/src/Assignment 05/outputs/metrics/concepts.json')